<a href="https://colab.research.google.com/github/Mohsin-22/Mohsin-22-Assignment4-100_Gen-Ai_cohort/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Assignment 4**: Build a Semantic Search Ready NLP Pipeline from Raw Text Data.

In [ ]:

!pip install nltk spacy gensim datasets scikit-learn matplotlib seaborn
!python -m spacy download en_core_web_sm


In [ ]:

import re
import nltk
import spacy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec


In [ ]:
nltk.download('stopwords')

In [ ]:
nlp_engine = spacy.load("en_core_web_sm")
stop_words_set = set(stopwords.words("english"))


In [ ]:
raw_dataset = load_dataset("amazon_polarity", split="train[:2000]")
feedback_texts = [item["content"] for item in raw_dataset]

In [ ]:
def clean_sentence(text_line):
    text_line = text_line.lower()
    text_line = re.sub(r"http\S+|www\S+", "", text_line)
    text_line = re.sub(r"[^a-z\s]", "", text_line)
    text_line = re.sub(r"\s+", " ", text_line)
    return text_line.strip()

cleaned_text_data = [clean_sentence(text) for text in feedback_texts]

In [ ]:
def preprocess_text(sentence):
    doc = nlp_engine(sentence)
    processed_tokens = [
        token.lemma_
        for token in doc
        if token.text not in stop_words_set
        and token.is_alpha
        and len(token.text) > 2
    ]
    return processed_tokens

In [ ]:
final_tokens_list = [preprocess_text(text) for text in cleaned_text_data]
final_sentences = [" ".join(tokens) for tokens in final_tokens_list]

In [ ]:
unique_vocab = set()
for tokens in final_tokens_list:
    unique_vocab.update(tokens)

print("Vocabulary size:", len(unique_vocab))

In [ ]:
bow_vectorizer = CountVectorizer(max_features=3000)
bow_matrix = bow_vectorizer.fit_transform(final_sentences)

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=3000)
tfidf_matrix = tfidf_vectorizer.fit_transform(final_sentences)

In [ ]:
word2vec_model = Word2Vec(
    sentences=final_tokens_list,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

In [ ]:
def create_sentence_vector(tokens, model):
    valid_vectors = [
        model.wv[word] for word in tokens if word in model.wv
    ]
    if len(valid_vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(valid_vectors, axis=0)

sentence_vectors = np.array([
    create_sentence_vector(tokens, word2vec_model)
    for tokens in final_tokens_list
])

In [ ]:
def find_similar_feedback(query_text, top_k=5):
    cleaned_query = clean_sentence(query_text)
    query_tokens = preprocess_text(cleaned_query)
    query_vector = create_sentence_vector(query_tokens, word2vec_model)

    similarity_scores = cosine_similarity(
        [query_vector],
        sentence_vectors
    )[0]

    top_indices = similarity_scores.argsort()[-top_k:][::-1]

    for idx in top_indices:
        print("Score:", round(similarity_scores[idx], 3))
        print("Text:", feedback_texts[idx][:200])
        print("-" * 80)

In [ ]:
find_similar_feedback("poor customer service and bad support response overall Worst experience")

In [ ]:
tfidf_mean_scores = np.array(tfidf_matrix.mean(axis=0)).flatten()
feature_names = tfidf_vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame({
    "word": feature_names,
    "avg_score": tfidf_mean_scores
}).sort_values(by="avg_score", ascending=False).head(15)

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(data=tfidf_df, x="avg_score", y="word")
plt.title("Top TF‑IDF Keywords in Reviews")
plt.show()